# Advanced VOI Methods Tutorial

This notebook walks through the practical advanced VOI methods that are
already planned in the `docs-developer-experience` track:

- structural VOI
- network meta-analysis (NMA) VOI
- adaptive trial VOI
- portfolio VOI
- sequential VOI
- efficient EVSI
- CEAF and extended dominance

The examples are intentionally small, deterministic, and executable so
they can serve as a practical starting point rather than a benchmark.

In [1]:
import numpy as np
import xarray as xr

from voiage.methods.adaptive import (
    adaptive_evsi,
    sophisticated_adaptive_trial_simulator,
)
from voiage.methods.ceaf import calculate_ceaf
from voiage.methods.dominance import (
    calculate_dominance,
    calculate_extended_dominance,
)
from voiage.methods.network_nma import evsi_nma
from voiage.methods.portfolio import portfolio_voi
from voiage.methods.sample_information import evsi
from voiage.methods.sequential import sequential_voi
from voiage.methods.structural import structural_evpi, structural_evppi
from voiage.schema import (
    DecisionOption,
    DynamicSpec,
    ParameterSet,
    PortfolioSpec,
    PortfolioStudy,
    TrialDesign,
    ValueArray,
)

np.random.seed(42)

def make_parameter_set(data: dict[str, np.ndarray]) -> ParameterSet:
    return ParameterSet.from_numpy_or_dict(data)

def make_value_array(values: np.ndarray, strategy_names: list[str]) -> ValueArray:
    return ValueArray.from_numpy(np.asarray(values, dtype=float), strategy_names)

## Structural VOI

Structural VOI compares alternative model structures and quantifies the value of learning which structure is correct.

In [2]:
def structural_model_a(psa: ParameterSet) -> ValueArray:
    severity = psa.parameters['severity']
    treatment_effect = psa.parameters['treatment_effect']
    net_benefits = np.column_stack([
        120.0 + 30.0 * treatment_effect - 8.0 * severity,
        110.0 + 35.0 * treatment_effect - 4.0 * severity,
        105.0 + 28.0 * treatment_effect - 2.0 * severity,
    ])
    return make_value_array(net_benefits, ['Standard care', 'Treat A', 'Treat B'])

def structural_model_b(psa: ParameterSet) -> ValueArray:
    severity = psa.parameters['severity']
    treatment_effect = psa.parameters['treatment_effect']
    net_benefits = np.column_stack([
        118.0 + 26.0 * treatment_effect - 10.0 * severity,
        112.0 + 32.0 * treatment_effect - 6.0 * severity,
        108.0 + 34.0 * treatment_effect - 3.0 * severity,
    ])
    return make_value_array(net_benefits, ['Standard care', 'Treat A', 'Treat B'])

structural_psa_a = make_parameter_set({
    'severity': np.linspace(0.1, 0.6, 40),
    'treatment_effect': np.linspace(0.2, 0.5, 40),
})
structural_psa_b = make_parameter_set({
    'severity': np.linspace(0.2, 0.7, 40),
    'treatment_effect': np.linspace(0.15, 0.45, 40),
})

structural_evpi_value = structural_evpi(
    [structural_model_a, structural_model_b],
    [0.55, 0.45],
    [structural_psa_a, structural_psa_b],
)
structural_evppi_value = structural_evppi(
    [structural_model_a, structural_model_b],
    [0.55, 0.45],
    [structural_psa_a, structural_psa_b],
    structures_of_interest=[1],
)

print(f'Structural EVPI: {structural_evpi_value:.3f}')
print(f'Structural EVPPI for structure 1: {structural_evppi_value:.3f}')

Structural EVPI: 0.000
Structural EVPPI for structure 1: 0.000


## NMA VOI

This example uses a tiny synthetic treatment network and a toy evaluator that maps posterior samples into net-benefit surfaces.

In [3]:
def toy_nma_evaluator(
    psa: ParameterSet,
    trial_design: TrialDesign | None,
    trial_data: object | None,
) -> ValueArray:
    baseline = psa.parameters['baseline']
    effect_b = psa.parameters['effect_b']
    effect_c = psa.parameters['effect_c']
    net_benefits = np.column_stack([
        baseline + 20.0,
        baseline + 60.0 * effect_b,
        baseline + 70.0 * effect_c,
    ])
    return make_value_array(net_benefits, ['Treatment A', 'Treatment B', 'Treatment C'])

nma_psa = make_parameter_set({
    'baseline': np.linspace(90.0, 110.0, 60),
    'effect_b': np.linspace(0.2, 0.5, 60),
    'effect_c': np.linspace(0.15, 0.6, 60),
})
nma_trial_design = TrialDesign([
    DecisionOption('Treatment A', 30),
    DecisionOption('Treatment B', 30),
    DecisionOption('Treatment C', 30),
])

nma_evsi_value = evsi_nma(
    toy_nma_evaluator,
    nma_psa,
    nma_trial_design,
    n_outer_loops=4,
    n_inner_loops=5,
)

print(f'NMA EVSI: {nma_evsi_value:.3f}')

NMA EVSI: 0.000


## Adaptive Trial VOI

The adaptive trial example reuses the package’s built-in adaptive trial simulator with a compact prior and simple interim rules.

In [4]:
adaptive_psa = make_parameter_set({
    'treatment_effect': np.linspace(0.05, 0.18, 80),
    'control_rate': np.linspace(0.20, 0.35, 80),
    'cost_per_patient': np.linspace(4500.0, 5500.0, 80),
})
adaptive_design = TrialDesign([
    DecisionOption('Control', 75),
    DecisionOption('Treatment', 75),
])
adaptive_rules = {
    'interim_analysis_points': [0.5],
    'early_stopping_rules': {'efficacy': 0.95, 'futility': 0.10},
    'sample_size_reestimation': True,
}

adaptive_evsi_value = adaptive_evsi(
    sophisticated_adaptive_trial_simulator,
    adaptive_psa,
    adaptive_design,
    adaptive_rules,
    n_outer_loops=3,
    n_inner_loops=5,
)

print(f'Adaptive trial EVSI: {adaptive_evsi_value:.3f}')

Adaptive trial EVSI: 0.000


## Portfolio VOI

Portfolio VOI selects the studies that are worth funding subject to a budget constraint.

In [5]:
portfolio_studies = [
    PortfolioStudy('Feasibility study', TrialDesign([DecisionOption('Arm A', 20), DecisionOption('Arm B', 20)]), 35000.0),
    PortfolioStudy('Confirmatory study', TrialDesign([DecisionOption('Arm A', 50), DecisionOption('Arm B', 50)]), 65000.0),
    PortfolioStudy('Implementation study', TrialDesign([DecisionOption('Arm A', 40), DecisionOption('Arm B', 40)]), 45000.0),
]
portfolio_spec = PortfolioSpec(portfolio_studies, budget_constraint=100000.0)

def study_value_calculator(study: PortfolioStudy) -> float:
    return 1500.0 * study.design.total_sample_size - 0.35 * study.cost

portfolio_result = portfolio_voi(
    portfolio_spec,
    study_value_calculator,
    optimization_method='dynamic_programming',
)

print('Selected studies:', [study.name for study in portfolio_result['selected_studies']])
print(f"Total portfolio value: {portfolio_result['total_value']:.3f}")
print(f"Total portfolio cost: {portfolio_result['total_cost']:.3f}")

Selected studies: ['Feasibility study', 'Confirmatory study']
Total portfolio value: 175000.000
Total portfolio cost: 100000.000


## Sequential VOI

Sequential VOI evaluates the value of waiting for more information over multiple time steps.

In [6]:
sequential_dataset = xr.Dataset(
    {
        'net_benefits': (
            ('n_samples', 'n_strategies'),
            np.array([
                [9.0, 11.0, 10.0],
                [10.0, 9.5, 11.5],
                [8.5, 10.0, 10.5],
                [9.5, 10.5, 11.0],
            ], dtype=float),
        )
    },
    coords={
        'n_samples': np.arange(4),
        'n_strategies': np.arange(3),
        'strategy': ('n_strategies', ['Observe', 'Treat now', 'Treat later']),
    },
)
sequential_psa = ParameterSet(dataset=sequential_dataset)
dynamic_spec = DynamicSpec([0.0, 1.0, 2.0])

def hold_state(psa: ParameterSet, action: object, dyn_spec: DynamicSpec) -> dict[str, object]:
    return {'next_psa': psa, 'action': action, 'time_steps': list(dyn_spec.time_steps)}

sequential_value = sequential_voi(
    hold_state,
    sequential_psa,
    dynamic_spec,
    wtp=50000.0,
    optimization_method='backward_induction',
)

print(f'Sequential VOI: {sequential_value:.3f}')

Sequential VOI: 0.750


## Efficient EVSI

Efficient EVSI uses the regression-backed approximation path instead of the slower nested Monte Carlo loop.

In [7]:
def evsi_model(psa: ParameterSet) -> ValueArray:
    biomarker = psa.parameters['biomarker']
    cost_shift = psa.parameters['cost_shift']
    net_benefits = np.column_stack([
        95.0 + 25.0 * biomarker - 0.20 * cost_shift,
        100.0 + 42.0 * biomarker - 0.15 * cost_shift,
    ])
    return make_value_array(net_benefits, ['Standard care', 'New test'])

evsi_psa = make_parameter_set({
    'biomarker': np.linspace(0.0, 1.0, 120),
    'cost_shift': np.linspace(0.0, 50.0, 120),
})
evsi_design = TrialDesign([
    DecisionOption('Standard care', 60),
    DecisionOption('New test', 60),
])

efficient_evsi_value = evsi(
    evsi_model,
    evsi_psa,
    evsi_design,
    method='efficient',
    metamodel='linear',
    n_outer_loops=5,
    n_inner_loops=5,
)

print(f'Efficient EVSI: {efficient_evsi_value:.3f}')

Efficient EVSI: 0.000


## CEAF and Extended Dominance

These helpers summarize the cost-effectiveness frontier and identify strategies removed by strong or extended dominance.

In [8]:
costs = np.array([100.0, 130.0, 180.0, 240.0])
effects = np.array([1.2, 1.3, 1.55, 1.7])
dominance_result = calculate_dominance(costs, effects, strategy_names=['A', 'B', 'C', 'D'])
extended_dominated = calculate_extended_dominance(costs, effects)

print('Dominance frontier:', dominance_result.frontier_indices)
print('Extended-dominated indices:', extended_dominated)

wtp_thresholds = np.array([0.0, 50000.0, 100000.0])
base_costs = np.array([
    [90.0, 120.0, 150.0],
    [95.0, 118.0, 148.0],
    [100.0, 125.0, 152.0],
    [92.0, 122.0, 149.0],
])
base_effects = np.array([
    [1.00, 1.05, 1.12],
    [1.02, 1.08, 1.14],
    [0.98, 1.04, 1.10],
    [1.01, 1.07, 1.15],
])
surface = np.stack([wtp * base_effects - base_costs for wtp in wtp_thresholds], axis=2)
ceaf_value_array = ValueArray(
    dataset=xr.Dataset(
        {'net_benefit': (('n_samples', 'n_strategies', 'n_wtp'), surface)},
        coords={
            'n_samples': np.arange(surface.shape[0]),
            'n_strategies': np.arange(surface.shape[1]),
            'n_wtp': np.arange(surface.shape[2]),
            'strategy': ('n_strategies', ['A', 'B', 'C']),
        },
    )
)
ceaf_result = calculate_ceaf(ceaf_value_array, wtp_thresholds.tolist())

print('CEAF optimal strategies:', ceaf_result.optimal_strategy_names)
print('CEAF acceptability probabilities:', np.round(ceaf_result.acceptability_probabilities, 3).tolist())

Dominance frontier: [0, 2, 3]
Extended-dominated indices: [1]
CEAF optimal strategies: ['A', 'C', 'C']
CEAF acceptability probabilities: [1.0, 1.0, 1.0]


## When To Use Which Method

- Structural VOI when the competing model structures themselves matter.
- NMA VOI when the study changes a treatment network rather than a single pairwise comparison.
- Adaptive trial VOI when interim looks and design adaptation are part of the study plan.
- Portfolio VOI when multiple studies compete for limited budget.
- Sequential VOI when decisions are staged over time.
- Efficient EVSI when you want a fast approximation rather than the exact nested loop.
- CEAF and extended dominance when you need a frontier view of strategy ordering and elimination.